In [2]:
pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 772.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 126.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 98.6 MB/s eta 0:00:00
  Attempting uninstall: h5py
    Found existing installation: h5py 3.16.0
    Uninstalling h5py-3.16.0:
      Successfully uninstalled h5py-3.16.0


In [3]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer

In [5]:
from datasets import load_dataset

dataset = load_dataset("roneneldan/TinyStories")

README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [6]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})


In [7]:
df = pd.DataFrame(dataset['train'])

In [8]:
df = df.head(10000)

In [9]:
print(f"Total rows: {len(df)}")

Total rows: 10000


In [10]:
tokenizer = Tokenizer()

In [11]:
tokenizer.fit_on_texts(df['text'])

In [12]:

len(tokenizer.word_index)

10251

In [13]:
df['tokenized'] = tokenizer.texts_to_sequences(df['text'])
print(df[['text', 'tokenized']].head(10))

                                                text  \
0  One day, a little girl named Lily found a need...   
1  Once upon a time, there was a little car named...   
2  One day, a little fish named Fin was swimming ...   
3  Once upon a time, in a land full of trees, the...   
4  Once upon a time, there was a little girl name...   
5  Once upon a time, in a big lake, there was a b...   
6  Once upon a time, in a small town, there was a...   
7  Once upon a time, in a peaceful town, there li...   
8  Once upon a time, there was a clever little do...   
9  One day, a fast driver named Tim went for a ri...   

                                           tokenized  
0  [25, 18, 4, 28, 41, 59, 20, 99, 4, 1827, 13, 1...  
1  [35, 40, 4, 29, 27, 5, 4, 28, 157, 59, 1809, 1...  
2  [25, 18, 4, 28, 296, 59, 1851, 5, 781, 422, 1,...  
3  [35, 40, 4, 29, 13, 4, 1106, 426, 19, 436, 27,...  
4  [35, 40, 4, 29, 27, 5, 4, 28, 41, 59, 20, 20, ...  
5  [35, 40, 4, 29, 13, 4, 34, 590, 27, 5, 4, 650,... 

In [14]:
input_sequences = df["tokenized"]

In [15]:
max_len = max([len(x) for x in df['tokenized']])

In [16]:
max_len

878

In [17]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
padded_input_seq = pad_sequences(input_sequences,maxlen = max_len, padding = "pre")

In [18]:
padded_input_seq

array([[   0,    0,    0, ...,    2,  408,   80],
       [   0,    0,    0, ...,  405,  292,  156],
       [   0,    0,    0, ...,  300,  137,   55],
       ...,
       [   0,    0,    0, ...,  115,   48,   52],
       [   0,    0,    0, ...,   30,  143,  305],
       [   0,    0,    0, ...,   13,    1, 1926]], dtype=int32)

In [19]:
X= padded_input_seq[:,:-1]

In [20]:
y = padded_input_seq[:,-1]

In [21]:
X.shape

(10000, 877)

In [22]:
y.shape

(10000,)

In [23]:
from tensorflow.keras.utils import to_categorical

In [24]:
# y = to_categorical(y, num_classes = 10252)

In [25]:
y

array([  80,  156,   55, ...,   52,  305, 1926], dtype=int32)

In [26]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [27]:
y.shape

(10000,)

In [28]:
model = Sequential()
model.add(Embedding(10252, 100, input_length=878))
model.add(LSTM(150))
model.add(Dense(10252, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [29]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [31]:
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [32]:
model.fit(X, y, epochs=50, )

Epoch 1/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 16s 37ms/step - accuracy: 0.0831 - loss: 6.6287
Epoch 2/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 11s 36ms/step - accuracy: 0.1349 - loss: 5.6033
Epoch 3/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 11s 37ms/step - accuracy: 0.1799 - loss: 5.2202
Epoch 4/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 12s 37ms/step - accuracy: 0.2124 - loss: 4.9040
Epoch 5/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 11s 37ms/step - accuracy: 0.2483 - loss: 4.6115
Epoch 6/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 12s 37ms/step - accuracy: 0.2776 - loss: 4.3164
Epoch 7/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 11s 37ms/step - accuracy: 0.3077 - loss: 4.0172
Epoch 8/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 11s 36ms/step - accuracy: 0.3348 - loss: 3.7449
Epoch 9/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 11s 37ms/step - accuracy: 0.3605 - loss: 3.4951
Epoch 10/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 12s 37ms/step - accuracy: 0.3893 - loss: 3.2587
Epoch 11/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 12s 37ms/step - accuracy: 0.4081 - loss: 3.0350
Epoch 12/50
313/313 ━━━━━━━━━━

In [36]:
text  = "tell"

for i in range(5):
  token_text = tokenizer.texts_to_sequences([text])

  padded_token_text = pad_sequences(token_text, maxlen=14, padding='pre')
  pos = np.argmax(model.predict(padded_token_text))

  for word,index in tokenizer.word_index.items():
      if index == pos:
          text = text + " " + word
          print(text)




1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
tell soldier
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
tell soldier better
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
tell soldier better together
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
tell soldier better together again
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
tell soldier better together again soon


In [ ]:
from google.colab import files


model.save('next_word_model.h5')

import pickle
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

files.download('next_word_model.h5')
files.download('tokenizer.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>